In [1]:
import pandas as pd
import os

In [2]:
# Function to calculate cost
def calculate_cost(row):
    # Base cost per km
    cost = row['Distance [km]'] * cost_factor_transportation
    
    # Add additional costs for Suez or Panama, divided by distance
    if pd.notna(row['Suez or Panama']) and row['Distance [km]'] > 0:
        if 'Suez' in row['Suez or Panama']:
            cost += cost_factor_suez / row['Distance [km]']
        elif 'Panama' in row['Suez or Panama']:
            cost += cost_factor_panama / row['Distance [km]']
    
    return cost

# Function to create the new dataframe based on the "From" column for liquefaction cost
def create_LNG_liquefaction_df(df, factor):
    # Step 1: Create a new dataframe with unique "From" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['From'].unique(), columns=['From'])
    
    # Step 2: Adjust the "From" column by removing "_LNG" and place the original "From" values in the "To" column
    unique_from_df['To'] = unique_from_df['From']
    unique_from_df['From'] = unique_from_df['From'].str.replace('_LNG_exp', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

# Function to create the new dataframe based on the "To" column for regasification cost
def create_LNG_regasification_df(df, factor):
    # Step 1: Create a new dataframe with unique "To" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['To'].unique(), columns=['To'])
    
    # Step 2: Adjust the "To" column by removing "_LNG" and place the original "To" values in the "From" column
    unique_from_df['From'] = unique_from_df['To']
    unique_from_df['To'] = unique_from_df['To'].str.replace('_LNG_imp', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

#Define a function to multiply the distance by a cost factor for the pipelines
def pipeline_transport_cost(df, cost_factor):
    df["Cost"] = df["distance [km]"] * cost_factor
    return df

# Function to create df_supply_demand_global
def create_supply_demand_df(df, supply=True):
    # Create the new dataframe with required columns
    df_supply_demand_global = pd.DataFrame({
        'Commodity': ['Methane'] * len(df),  # Set 'Methane' for all rows
        'Node': df['Country'] + ('_Prod' if supply else ''),  # Append '_Prod' if supply is True
        'Supply': df['GWh [2020]']  # Use 'GWh [2020]' for supply
    })
    
    return df_supply_demand_global

def expand_with_hydrogen(df, hydrogen_investment):
    if hydrogen_investment:
        return df  # If investment is allowed, return the original dataframe unchanged

    # Check if the input dataframe follows the (Commodity, Source, Destination) structure
    if {'Commodity', 'Source', 'Destination'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Source', 'Destination']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
        
    # If the input dataframe follows the (Commodity, Node, Supply) structure
    elif {'Commodity', 'Node', 'Supply'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Node']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
    
    else:
        raise ValueError("Unexpected dataframe format. Must contain either ['Commodity', 'Source', 'Destination'] or ['Commodity', 'Node', 'Supply']")

    # Fill all other columns with 0
    for col in df.columns:
        if col not in hydrogen_df.columns:  # Skip the required columns
            hydrogen_df[col] = 0

    # Combine the original dataframe with the new hydrogen dataframe
    df_expanded = pd.concat([df, hydrogen_df], ignore_index=True)

    return df_expanded

def integrate_pipeline_costs(df_pipelines, df_transport_cost):
    """
    Merges transport cost data into the pipeline dataframe based on matching From-To relationships, 
    considering both directions (From->To and To->From).
    
    Parameters:
        df_pipelines (pd.DataFrame): DataFrame containing pipeline capacities.
        df_transport_cost (pd.DataFrame): DataFrame containing transport distances and costs.
        
    Returns:
        pd.DataFrame: Updated pipeline DataFrame with an additional 'cost' column.
    """
    # Create reversed pairs for bidirectional matching (From -> To and To -> From)
    df_reversed = df_transport_cost.rename(columns={'From': 'To', 'To': 'From', 'Cost': 'Cost_reversed'})
    
    # Concatenate original and reversed cost data to handle both directions
    df_cost = pd.concat([df_transport_cost[['From', 'To', 'Cost']], df_reversed[['From', 'To', 'Cost_reversed']]], ignore_index=True)
    
    # Merge the concatenated cost data with df_pipelines to get the corresponding cost
    df_pipelines = df_pipelines.merge(
        df_cost, 
        on=['From', 'To'], 
        how='left'
    )
    
    # For cases where the reverse relation exists, use the reversed cost value
    df_pipelines['Cost'] = df_pipelines['Cost'].fillna(df_pipelines['Cost_reversed'])

    # Drop the extra reversed cost column (no longer needed)
    df_pipelines = df_pipelines.drop(columns=['Cost_reversed'])
    
    return df_pipelines

def extract_unique_nodes(df):
    unique_nodes = pd.unique(df[['Source', 'Destination']].values.ravel())
    return pd.DataFrame({'Nodes': unique_nodes})

In [3]:
def create_base_edges(df):
    """Create a dataframe with unique (From, To) pairs."""
    unique_pairs = set((row['From'], row['To']) for _, row in df.iterrows())
    return pd.DataFrame(unique_pairs, columns=['Source', 'Destination'])

#function to generate structure of European pipeline parameters
def process_european_pipeline_edges(df_pipelines_europe):
    """Process European pipeline transport cost and capacity data."""
    df_pipelines = create_base_edges(df_pipelines_europe)

    # Merge cost & capacity
    df_pipelines = df_pipelines.merge(
        df_pipelines_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_pipelines'})

    # Default values for missing capacities
    df_pipelines['initial_capacities'] = df_pipelines['Capacity_pipelines'].fillna(9999999)
    df_pipelines['max_capacities'] = df_pipelines['initial_capacities']
    df_pipelines = df_pipelines.drop(columns=['Capacity_pipelines'])

    # Add default columns
    df_pipelines.insert(0, 'Commodity', 'Methane')
    df_pipelines['new_build_cost'] = 1000000
    df_pipelines['conversion_cost'] = 0
    df_pipelines['conversion_capacity_factor'] = 1

    column_to_move = df_pipelines.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_pipelines.insert(5, "costs_edge", column_to_move)

    return df_pipelines

#process LNG import function (for European import capacities)
def process_LNG_import_edges(df_LNG_europe):
    """Process LNG import terminal costs and capacities."""
    df_LNG = create_base_edges(df_LNG_europe)

    df_LNG = df_LNG.merge(
        df_LNG_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_LNG_import'})

    df_LNG['initial_capacities'] = df_LNG['Capacity_LNG_import'].fillna(9999999)
    df_LNG['max_capacities'] = df_LNG['initial_capacities']
    df_LNG = df_LNG.drop(columns=['Capacity_LNG_import'])

    # Add default columns
    df_LNG.insert(0, 'Commodity', 'Methane')
    df_LNG['new_build_cost'] = 1000000
    df_LNG['conversion_cost'] = 0
    df_LNG['conversion_capacity_factor'] = 1

    column_to_move = df_LNG.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_LNG.insert(5, "costs_edge", column_to_move)

    return df_LNG

#function to process LNG global exchange parameters for the edges
def process_LNG_global_edges(df_LNG_global):
    """Process global LNG transport costs and capacities."""
    df_LNG_global_processed = create_base_edges(df_LNG_global)

    # Merge cost and distance data
    df_LNG_global_processed = df_LNG_global_processed.merge(
        df_LNG_global[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_LNG_global'})

    # Handle capacity values
    df_LNG_global_processed['initial_capacities'] = df_LNG_global_processed['Capacity_LNG_global'].fillna(9999999)
    df_LNG_global_processed['max_capacities'] = df_LNG_global_processed['initial_capacities']
    df_LNG_global_processed = df_LNG_global_processed.drop(columns=['Capacity_LNG_global'])

    # Add default columns
    df_LNG_global_processed.insert(0, 'Commodity', 'Methane')
    df_LNG_global_processed['new_build_cost'] = 1000000
    df_LNG_global_processed['conversion_cost'] = 0
    df_LNG_global_processed['conversion_capacity_factor'] = 1

    # Move `costs_edge` column to the correct position
    column_to_move = df_LNG_global_processed.pop("costs_edge")
    df_LNG_global_processed.insert(5, "costs_edge", column_to_move)

    return df_LNG_global_processed


def process_LNG_regasification_edges(df_LNG_regasification):
    """Process LNG regasification costs."""
    df_regas = create_base_edges(df_LNG_regasification)

    df_regas = df_regas.merge(
        df_LNG_regasification[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge'})

        # Default values for missing capacities
    df_regas['initial_capacities'] = 9999999
    df_regas['max_capacities'] = df_regas['initial_capacities']

    # Add default columns
    df_regas.insert(0, 'Commodity', 'Methane')
    df_regas['new_build_cost'] = 1000000
    df_regas['conversion_cost'] = 0
    df_regas['conversion_capacity_factor'] = 1

    column_to_move = df_regas.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_regas.insert(5, "costs_edge", column_to_move)
    
    return df_regas


#function to create parameters for liquification nodes
def process_LNG_liquefaction_edges(df_LNG_liquefaction):
    """Process LNG liquefaction costs."""
    df_liquefaction = create_base_edges(df_LNG_liquefaction)

    df_liquefaction = df_liquefaction.merge(
        df_LNG_liquefaction[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge'})

    # Default values for missing capacities
    df_liquefaction['initial_capacities'] = 9999999
    df_liquefaction['max_capacities'] = df_liquefaction['initial_capacities']

    # Add default columns
    df_liquefaction.insert(0, 'Commodity', 'Methane')
    df_liquefaction['new_build_cost'] = 1000000
    df_liquefaction['conversion_cost'] = 0
    df_liquefaction['conversion_capacity_factor'] = 1

    column_to_move = df_liquefaction.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_liquefaction.insert(5, "costs_edge", column_to_move)

    return df_liquefaction

#function to create parameters for global pipelines
def process_global_pipeline_edges(df_global_pipe_transport_cost):
    """Process global pipeline transport costs."""
    df_global_pipeline = create_base_edges(df_global_pipe_transport_cost)

    df_global_pipeline = df_global_pipeline.merge(
        df_global_pipe_transport_cost[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'costs_edge', 'GWh/a': 'Capacity_pipelines'})

    # Default values for missing capacities
    df_global_pipeline['initial_capacities'] = df_global_pipeline['Capacity_pipelines'].fillna(9999)
    df_global_pipeline['max_capacities'] = df_global_pipeline['initial_capacities']
    df_global_pipeline = df_global_pipeline.drop(columns=['Capacity_pipelines'])

    # Add default columns
    df_global_pipeline.insert(0, 'Commodity', 'Methane')
    df_global_pipeline['new_build_cost'] = 1000000
    df_global_pipeline['conversion_cost'] = 0
    df_global_pipeline['conversion_capacity_factor'] = 1

    column_to_move = df_global_pipeline.pop("costs_edge")

    # insert column with insert(location, column_name, column_value)
    df_global_pipeline.insert(5, "costs_edge", column_to_move)

    return df_global_pipeline

def remove_existing_LNG_edges(df_regasification_LNG_edges, df_europe_LNG_edges):
    """Removes rows from df_regasification_LNG_edges if the Source-Destination pair exists in df_europe_LNG_edges."""
    existing_edges = set(zip(df_europe_LNG_edges['Source'], df_europe_LNG_edges['Destination']))
    
    df_filtered = df_regasification_LNG_edges[
        ~df_regasification_LNG_edges.apply(lambda row: (row['Source'], row['Destination']) in existing_edges, axis=1)
    ]
    
    return df_filtered

#function to create connections from missing Countries to LNG export connections
def ensure_direct_connections(df, df_LNG_europe, cost_factor_liquefaction):
    # Ensure no NaN values in 'Source' and create a new dataframe
    df_cleaned = df.dropna(subset=["Source"]).copy()

    # Extract all LNG sources from df_edges_cap_cost
    lng_sources = set(df_cleaned[df_cleaned["Source"].astype(str).str.endswith("_LNG")]["Source"])

    # Extract LNG sources that should be **excluded** (those in df_LNG_europe["From"])
    excluded_lng_sources = set(df_LNG_europe["From"].dropna().unique())

    # Keep only LNG sources that **should be considered**
    valid_lng_sources = lng_sources - excluded_lng_sources

    # Extract base country codes that need new connections
    base_countries = {src.replace("_LNG", "") for src in valid_lng_sources}

    # Initialize a list to collect missing rows
    missing_rows = []

    # Create the missing connections for the countries in base_countries
    for country in base_countries:
        source = country
        destination = f"{country}_LNG"

        # Ensure this direct connection doesn't already exist
        if not ((df_cleaned["Source"] == source) & (df_cleaned["Destination"] == destination)).any():
            missing_rows.append({
                "Commodity": "Methane",
                "Source": source,
                "Destination": destination,
                "initial_capacities": 9999999,
                "max_capacities": 9999999,
                "costs_edge": cost_factor_liquefaction,
                "new_build_cost": 1000000,
                "conversion_cost": 0,
                "conversion_capacity_factor": 1
            })

    # Create a new dataframe for the missing rows and return it
    df_missing = pd.DataFrame(missing_rows)

    return df_missing

#create edges for the production nodes of each country
def create_Production_node_edges(df_demand_supply_complete, production_cost_df):
    # Ensure the 'Node' column is treated as a string
    df_demand_supply_complete['Node'] = df_demand_supply_complete['Node'].astype(str)
    
    # Merge the supply-demand dataframe with the production cost dataframe on the 'Node' column
    df_with_costs = pd.merge(df_demand_supply_complete, production_cost_df, left_on='Node', right_on='Node', how='left')

    # Initialize an empty list to collect the rows for the new dataframe
    new_rows = []

    # Loop through the rows in the merged dataframe
    for _, row in df_with_costs.iterrows():
        node = row['Node']

        # Check if the node ends with "_Prod"
        if node.endswith("_Prod"):
            source = node
            destination = node.replace("_Prod", "")  # Remove "_Prod" from the node to get the destination

            # Get the production cost for this node from the merged dataframe
            production_cost = row['Cost']  # Assuming the production cost is in the 'Cost' column

            # Create a new row for the production connection with the known dummy values
            new_rows.append({
                "Commodity": row['Commodity'],  # Assuming it's "Methane" from the example
                "Source": source,
                "Destination": destination,
                "initial_capacities": 9999999,  # Dummy value
                "max_capacities": 9999999,  # Dummy value
                "costs_edge": production_cost,  # Use the production cost from the dataframe
                "new_build_cost": 1000000,  # Dummy value
                "conversion_cost": 0,  # Dummy value
                "conversion_capacity_factor": 1  # Dummy value
            })

    # Create a new dataframe from the collected rows
    df_Prod_edges = pd.DataFrame(new_rows)

    return df_Prod_edges

#function to remove countries that are not part of the analyses/model
def remove_rows_containing_strings(df, string_list):
    return df[
        ~df.apply(
            lambda row: row.astype(str)
                        .str.contains('|'.join(string_list), case=False, na=False)
                        .any(), 
            axis=1
        )
    ]

#add missing information to the demand and supply information but with value 0
def create_missing_prod_and_lng_nodes(df_demand_supply_complete, LNG_countries_list):
    # Step 1: Create missing _Prod nodes (but not for nodes already having _LNG)
    new_prod_nodes = []
    for node in df_demand_supply_complete['Node']:
        if not node.endswith("_Prod") and not node.endswith("_LNG"):
            prod_node = f"{node}_Prod"
            if prod_node not in df_demand_supply_complete['Node'].values:
                new_prod_nodes.append({
                    "Commodity": "Methane",
                    "Node": prod_node,
                    "Supply": 0
                })
    
    # Step 2: Create missing _LNG_export nodes
    new_lng_export_nodes = []
    for node in df_demand_supply_complete['Node']:
        country_code = node.split('_')[0]
        if not node.endswith("_Prod") and not node.endswith("_LNG_exp") and not node.endswith("_LNG_imp") and country_code in LNG_countries_list:
            new_lng_export_nodes.append({
                "Commodity": "Methane",
                "Node": f"{node}_LNG_exp",
                "Supply": 0
            })

    # Step 2: Create missing _LNG_import nodes
    new_lng_import_nodes = []
    for node in df_demand_supply_complete['Node']:
        country_code = node.split('_')[0]
        if not node.endswith("_Prod") and not node.endswith("_LNG_exp") and not node.endswith("_LNG_imp") and country_code in LNG_countries_list:
            new_lng_import_nodes.append({
                "Commodity": "Methane",
                "Node": f"{node}_LNG_imp",
                "Supply": 0
            })
    
    # Convert new nodes lists to DataFrames
    df_new_prod_nodes = pd.DataFrame(new_prod_nodes)
    df_new_lng_export_nodes = pd.DataFrame(new_lng_export_nodes)
    df_new_lng_import_nodes = pd.DataFrame(new_lng_import_nodes)

    # Combine both DataFrames
    df_new_nodes = pd.concat([df_new_prod_nodes, df_new_lng_export_nodes, df_new_lng_import_nodes], ignore_index=True)

    # Return new nodes dataframe
    return df_new_nodes

#combine all df for the supply inpput sheet (demand (negative) or supply (positive) values
def add_missing_supply_nodes(df_demand_supply_complete, df_missing_supply_values_for_nodes):
    # Merge on 'Commodity' and 'Node', keeping existing values in df_demand_supply_complete
    df_combined = pd.concat([df_demand_supply_complete, df_missing_supply_values_for_nodes]) \
                    .drop_duplicates(subset=['Commodity', 'Node'], keep='first') \
                    .reset_index(drop=True)
    return df_combined

def extract_unique_commodities(df_demand_supply_complete):
    unique_commodities = df_demand_supply_complete['Commodity'].unique()
    return pd.DataFrame({'Commodities': unique_commodities})

def merge_all_edges(*dfs):
    """
    Merges multiple edge dataframes while removing duplicate Source-Destination pairs.
    
    Args:
        *dfs: Any number of dataframes to be merged.
    
    Returns:
        A merged dataframe with duplicates removed.
    """
    # Step 1: Concatenate all dataframes
    df_merged = pd.concat(dfs, ignore_index=True)

    # Step 2: Remove duplicate edges (keeping the first occurrence)
    df_merged = df_merged.drop_duplicates(subset=['Source', 'Destination'], keep='first')

    return df_merged

def update_parameter(df_edges_complete, df_updates, parameter_name, update_column):
    """
    Updates the specified parameter in df_edges_complete based on df_updates.

    Parameters:
    df_edges_complete (pd.DataFrame): Original dataframe with edges and costs.
    df_updates (pd.DataFrame): DataFrame containing updates with 
                               Commodity, Source, Destination, and new values.
    parameter_name (str): The name of the column to update in df_edges_complete.
    update_column (str): The name of the column in df_updates that contains new values.

    Returns:
    pd.DataFrame: Updated df_edges_complete with modified parameter values.
    """
    # Rename the update column to match parameter_name for easier merging
    df_updates = df_updates.rename(columns={update_column: parameter_name})
    
    # Merge the two dataframes on Commodity, Source, and Destination
    df_updated = df_edges_complete.merge(
        df_updates, 
        on=["Commodity", "Source", "Destination"], 
        how="left", 
        suffixes=("", "_update")
    )
    
    # Update the specified parameter where new values exist
    df_updated[parameter_name] = df_updated[f"{parameter_name}_update"].combine_first(df_updated[parameter_name])
    
    # Drop the temporary column
    df_updated.drop(columns=[f"{parameter_name}_update"], inplace=True)
    
    return df_updated

In [4]:
#to define later values for hydrogen if not provided (all 0 but necessary). 
hydrogen_investment=False

In [5]:
#set szenario
szenario_year = '2024'
szenario_name = 'run_' + szenario_year

In [6]:
#output specifications
output_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed', '01_paper_IAEE')
outout_file_name = '\inputs_IAEE_2025_' + szenario_name + '22.xlsx'
#create full ouput paths
output_file_path_excel  = output_file_path + outout_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path_excel))

## Import Data

In [7]:
# Specify the path to your Excel file
input_file_path_1 = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = '\\Gas_Import_Russian_Invasion.xlsx'
LNG_cost_file_name = '\\LNG_Transportation_Cost_Calculation.xlsx'
# Specify the path to your Excel file
input_file_path_2 = os.path.join('..', '..','00_code_base', '07_data_prep')
distances_file_name = '\\distances_' + szenario_year + '.xlsx'
# Specify the path to your Excel file
input_file_path_3 = os.path.join('..', '..','01_data', '01_input_data', '01_raw', '01_Russian_War_Case')
szenarios_update_file_name = '\\Szenarios_update_information.xlsx'

#create full input paths
input_file_path_excel  = input_file_path_1 + excel_file_name
full_input_path_1 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_excel))
input_file_path_LNG = input_file_path_1 + LNG_cost_file_name
full_input_path_LNG_cost = os.path.abspath(os.path.join(os.getcwd(), input_file_path_LNG))
input_file_path_2  = input_file_path_2 + distances_file_name
full_input_path_2 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_2))
full_input_path_3 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_3 + szenarios_update_file_name))

In [8]:
df_LNG_global = pd.read_excel(full_input_path_1, sheet_name='global_LNG_connections_' + szenario_year)
df_LNG_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_lng_europe_' + szenario_year)
df_production_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_europe_' + szenario_year)
df_consumption_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_europe_2035')
df_production_global = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_world_' + szenario_year)
df_consumption_global = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_world_2035')
df_pipelines_europe = pd.read_excel(full_input_path_1, sheet_name='Connections_' + szenario_year)
df_pipelines_global = pd.read_excel(full_input_path_1, sheet_name='Global_Connections')

df_updates = pd.read_excel(full_input_path_3, sheet_name = szenario_year)
#only relevant for the investment case
df_updates_invest = pd.read_excel(full_input_path_3, sheet_name = szenario_year + '_InvesPipes')

#load LNG and production cost data 
df_LNG_cost = pd.read_excel(full_input_path_LNG_cost, sheet_name='LNG_Cost_Params')
df_production_cost = pd.read_excel(full_input_path_LNG_cost, sheet_name='Production_Cost_Nodes')
#to be alligned in the future
production_cost_df = df_production_cost

#load input for inner-European distances
df_distances = pd.read_excel(full_input_path_2, sheet_name='distances')
#Drop the "Unnamed: 0" column in df_distances
df_distances.drop(columns=["Unnamed: 0"], inplace=True)

In [10]:
#df_consumption_global
df_consumption_europe

,Country,Long_name,Mrd m3 [2035],Mrd m3 [2035] AP,Mrd m3 [2023],TWh [2023],Population 2020 [Tsd],GWh [2020],GWh [2035],GWh [2035] AP,TWh by share,EU,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,AL,Albania,1.002185,0.761660,NaN,0.000000,2846,-9.856352e+03,-9791.345574,-7441.422636,-9.856352,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AT,Austria,5.538198,3.598751,6.881279,-67.230092,8901.1,-6.723009e+04,-54108.195930,-35159.800468,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BE,Belgium,11.030988,7.167996,13.706137,-133.908962,11549.9,-1.339090e+05,-107772.756819,-70031.324470,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BA,Bosnia and Herzegovina,1.229666,0.934546,NaN,0.000000,3492,-1.209360e+04,-12013.836523,-9130.515758,-12.093598,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BG,Bulgaria,2.032978,1.321040,2.526000,-24.679020,6951.5,-2.467902e+04,-19862.195856,-12906.563066,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,HR,Croatia,2.012168,1.307518,2.500144,-24.426404,4058.2,-2.442640e+04,-19658.885114,-12774.450639,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,CZ,Czech Republic,5.390289,3.502639,6.697500,-65.434575,10693.9,-6.543457e+04,-52663.126187,-34220.786277,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,DK,Denmark,1.312060,0.852584,1.630251,-15.927552,5822.8,-1.592755e+04,-12818.830044,-8329.745593,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,EE,Estonia,0.297691,0.193441,0.369884,-3.613770,1329,-3.613770e+03,-2908.438609,-1889.919251,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,FI,Finland,0.961292,0.624653,1.194417,-11.669451,5525.3,-1.166945e+04,-9391.820177,-6102.855913,0.000000,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [211]:
#adjust the data frames and remove unnecessary content
#gobal demand
df_consumption_global = df_consumption_global.iloc[:-3]
#gobal production
df_production_global = df_production_global.iloc[:-4]
#globale pipelines
df_pipelines_global = df_pipelines_global.iloc[:, :-5]
#europe demand
df_consumption_europe = df_consumption_europe.iloc[:-11]
#europe production
df_production_europe = df_production_europe.iloc[:-7]
df_production_europe = df_production_europe.iloc[:, :-7]
#europe LNG
df_LNG_europe = df_LNG_europe.iloc[:, :-6]
#add a From column to the European LNG data
df_LNG_europe.insert(1, 'From', df_LNG_europe['To'] + '_LNG')
#LNG global
df_LNG_global = df_LNG_global.iloc[:, :-5]

#adjust naming
df_distances = df_distances.rename(columns={"from [NUTS_ID]": "From"})
df_distances = df_distances.rename(columns={"to [NUTS_ID]": "To"})

In [212]:
#to remove if in the df as not part of the gas network (no data for interconnectors) 
regions_to_remove_list = ['Other Europe', 'San Marino', 'Malta', 'Kosovo', 'Zyprus', 'Montenegro']
df_consumption_europe = remove_rows_containing_strings(df_consumption_europe, regions_to_remove_list)
df_production_europe = remove_rows_containing_strings(df_production_europe, regions_to_remove_list)
df_LNG_europe = remove_rows_containing_strings(df_LNG_europe, regions_to_remove_list)

In [213]:
#set values for parameters
cost_factor_transportation = df_LNG_cost.loc[df_LNG_cost['type'] == 'cost per GWh and km', 'Cost'].values[0]
cost_factor_liquefaction = df_LNG_cost.loc[df_LNG_cost['type'] == 'liquefication per GWh', 'Cost'].values[0]
cost_factor_regasification = df_LNG_cost.loc[df_LNG_cost['type'] == 'regasification per GWh', 'Cost'].values[0]
cost_factor_panama = df_LNG_cost.loc[df_LNG_cost['type'] == 'Cost Panama per GWh', 'Cost'].values[0]
cost_factor_suez = df_LNG_cost.loc[df_LNG_cost['type'] == 'Cost Suez per GWh', 'Cost'].values[0]

cost_factor_pipelines = df_LNG_cost.loc[df_LNG_cost['type'] == 'cost pipe per GWh and km', 'Cost'].values[0]

### Demand and Supply input sheet

In [214]:
#demand Europe
df_demand_europe = create_supply_demand_df(df_consumption_europe, supply=False)
#supply Europe
df_supply_europe = create_supply_demand_df(df_production_europe)

In [215]:
#demand Europe
df_demand_global = create_supply_demand_df(df_consumption_global, supply=False)
#supply global
df_supply_global = create_supply_demand_df(df_production_global)

In [216]:
# Combine the demand and supply data frames
df_supply_demand = pd.concat([df_demand_europe, df_supply_europe, df_demand_global, df_supply_global], ignore_index=True)

In [217]:
# Apply the function to each row and create a new column 'Cost per km'
df_LNG_global['Cost'] = df_LNG_global.apply(calculate_cost, axis=1)

# Create a new dataframe with the relevant columns
df_cost_per_km = df_LNG_global[['From', 'To', 'Distance [km]', 'Suez or Panama', 'Cost']]

In [218]:
#add export and import information
df_LNG_global['From'] = df_LNG_global['From'].str.replace('_LNG', '_LNG_exp')
df_LNG_global['To'] = df_LNG_global['To'].str.replace('_LNG', '_LNG_imp')

In [219]:
# Get LNG liquefaction cost df
LNG_liquefaction_df = create_LNG_liquefaction_df(df_LNG_global, cost_factor_liquefaction)
# Get LNG regasification cost df
LNG_regasification_df = create_LNG_regasification_df(df_LNG_global, cost_factor_regasification)

In [220]:
#add regasification cost to European LNG import nodes
df_LNG_europe.insert(3, "Cost", cost_factor_regasification)
#add export and import information
df_LNG_europe['From'] = df_LNG_europe['From'].str.replace('_LNG', '_LNG_imp')
df_LNG_europe['To'] = df_LNG_europe['To'].str.replace('_LNG', '_LNG_exp')

In [221]:
#Pipeline cost
#calculate European pipeline cost
df_european_pipe_transport_cost = pipeline_transport_cost(df_distances, cost_factor_pipelines)
df_global_pipe_transport_cost = pipeline_transport_cost(df_pipelines_global, cost_factor_pipelines)

In [222]:
# Example usage:
df_pipelines_europe = integrate_pipeline_costs(df_pipelines_europe, df_european_pipe_transport_cost)

In [223]:
df_europe_pipe_edges = process_european_pipeline_edges(df_pipelines_europe)

In [224]:
df_global_pipe_edges = process_global_pipeline_edges(df_global_pipe_transport_cost)
df_liquefaction_LNG_edges = process_LNG_liquefaction_edges(LNG_liquefaction_df)
df_regasification_LNG_edges = process_LNG_regasification_edges(LNG_regasification_df)
df_europe_LNG_edges = process_LNG_import_edges(df_LNG_europe)
df_europe_pipe_edges = process_european_pipeline_edges(df_pipelines_europe)
df_LNG_global_edges = process_LNG_global_edges(df_LNG_global)
#reomve duplicates from the regasification edges df
df_regasification_LNG_edges = remove_existing_LNG_edges(df_regasification_LNG_edges, df_europe_LNG_edges)
#add missing edges for the production
df_production_edges = create_Production_node_edges(df_supply_demand, production_cost_df)
#add missing edges for LNG exporting countries
df_missing_LNG_export_edges = ensure_direct_connections(df_LNG_global_edges, df_LNG_europe, cost_factor_liquefaction)

In [225]:
# Apply the create_edges_cap_cost_dataframe function to get edges input 
df_edges_cap_cost = merge_all_edges(
    df_europe_pipe_edges, 
    df_europe_LNG_edges, 
    df_regasification_LNG_edges, 
    df_liquefaction_LNG_edges, 
    df_global_pipe_edges, 
    df_LNG_global_edges,
    df_missing_LNG_export_edges,
    df_production_edges
)

In [226]:
df_europe_LNG_edges

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,NL_LNG_imp,NL,117240.0,117240.0,2217.891962,1000000,0,1
1,Methane,ES_LNG_imp,ES,587177.0,587177.0,2217.891962,1000000,0,1
2,Methane,PL_LNG_imp,PL,56666.0,56666.0,2217.891962,1000000,0,1
3,Methane,HR_LNG_imp,HR,25402.0,25402.0,2217.891962,1000000,0,1
4,Methane,IT_LNG_imp,IT,154366.0,154366.0,2217.891962,1000000,0,1
5,Methane,FR_LNG_imp,FR,322410.0,322410.0,2217.891962,1000000,0,1
6,Methane,UK_LNG_imp,UK,469937.0,469937.0,2217.891962,1000000,0,1
7,Methane,EL_LNG_imp,EL,68390.0,68390.0,2217.891962,1000000,0,1
8,Methane,BE_LNG_imp,BE,87930.0,87930.0,2217.891962,1000000,0,1
9,Methane,TR_LNG_imp,TR,144596.0,144596.0,2217.891962,1000000,0,1


In [227]:
df_LNG_global_edges

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,IM_LNG_exp,EL_LNG_imp,9999999.0,9999999.0,872.844755,1000000,0,1
1,Methane,USA_LNG_exp,UK_LNG_imp,9999999.0,9999999.0,604.465606,1000000,0,1
2,Methane,EG_LNG_exp,BE_LNG_imp,9999999.0,9999999.0,415.586297,1000000,0,1
3,Methane,IM_LNG_exp,AF_LNG_imp,9999999.0,9999999.0,1082.234392,1000000,0,1
4,Methane,IM_LNG_exp,FR_LNG_imp,9999999.0,9999999.0,1152.644680,1000000,0,1
...,...,...,...,...,...,...,...,...,...
164,Methane,EG_LNG_exp,TR_LNG_imp,9999999.0,9999999.0,93.532826,1000000,0,1
165,Methane,TT_LNG_exp,EG_LNG_imp,9999999.0,9999999.0,711.708316,1000000,0,1
166,Methane,USA_LNG_exp,LT_LNG_imp,9999999.0,9999999.0,729.866958,1000000,0,1
167,Methane,AF_LNG_exp,IN_LNG_imp,9999999.0,9999999.0,910.714361,1000000,0,1


### check for hydrogen and append with zeros if no repurpose investigation

In [228]:
# Example: Calling function with hydrogen_investment = False
df_edges_complete = expand_with_hydrogen(df_edges_cap_cost, hydrogen_investment)

In [229]:
# Extract country codes matching the pattern from Source
source_codes = df_edges_complete['Source'].str.extract(r'([A-Z]{2,3})_LNG_imp')[0].dropna().unique()

# Extract country codes matching the pattern from Destination
dest_codes = df_edges_complete['Destination'].str.extract(r'([A-Z]{2,3})_LNG_exp')[0].dropna().unique()

# Combine and remove duplicates
LNG_countries = set(source_codes) | set(dest_codes)

# Convert to sorted list
LNG_countries_list = sorted(LNG_countries)

In [230]:
#move to later step
df_network_nodes = extract_unique_nodes(df_edges_complete)

In [231]:
df_missing_supply_values_for_nodes  = create_missing_prod_and_lng_nodes(df_supply_demand, LNG_countries_list)

In [232]:
df_all_supply_demand = add_missing_supply_nodes(df_supply_demand, df_missing_supply_values_for_nodes)

In [233]:
# Example: Calling function with hydrogen_investment = False
df_demand_supply_complete = expand_with_hydrogen(df_all_supply_demand, hydrogen_investment)

# Display the result
df_demand_supply_complete

,Commodity,Node,Supply
0,Methane,AL,-7717.933443
1,Methane,AT,-83311.215033
2,Methane,BE,-166072.218600
3,Methane,BA,-9469.790436
4,Methane,BG,-28534.750500
...,...,...,...
323,Hydrogen,IN_LNG_imp,0.000000
324,Hydrogen,AS_LNG_imp,0.000000
325,Hydrogen,RU_LNG_imp,0.000000
326,Hydrogen,EG_LNG_imp,0.000000


In [234]:
#get all commodities of the model
df_commodities = extract_unique_commodities(df_demand_supply_complete)

#get separate df for the edges
df_edges_only = df_edges_cap_cost[['Source', 'Destination']]
#probably it is df_edges_cap_cost as now all edges are defined twice

### update relevant parameters for analysis

In [235]:
# Update the costs_edge parameter for Russia to Europe to avoid the use of Russian pipeline gas
df_edges_complete = update_parameter(df_edges_complete, df_updates, "costs_edge", "costs_new")

In [237]:
#only relevant in the invest case to update investment cost
df_edges_complete = update_parameter(df_edges_complete, df_updates_invest, "new_build_cost", "costs_new")
#only relevant in the invest case to update investment limit
df_edges_complete = update_parameter(df_edges_complete, df_updates_invest , "max_capacities", "limit_new")

In [239]:
df_updates_invest

,Commodity,Source,Destination,costs_new,limit_new
0,Methane,AL,IT,1,1000000
1,Methane,AT,DE,1,1000000
2,Methane,AT,HU,1,1000000
3,Methane,AT,IT,1,1000000
4,Methane,AT,SI,1,1000000
...,...,...,...,...,...
103,Methane,UA,RO,1,1000000
104,Methane,UA,SK,1,1000000
105,Methane,UK,BE,1,1000000
106,Methane,UK,IE,1,1000000


# export

In [238]:
# Export to Excel with multiple sheets
with pd.ExcelWriter(full_output_file_path, engine='openpyxl') as writer:
    df_network_nodes.to_excel(writer, index=False, sheet_name='Nodes')
    df_commodities.to_excel(writer, index=False, sheet_name='Commodities')
    df_edges_only.to_excel(writer, index=False, sheet_name='Edges')
    df_edges_complete.to_excel(writer, index=False, sheet_name='Parameters')
    df_demand_supply_complete.to_excel(writer, index=False, sheet_name='Supply')